In [1]:
import numpy as np
import pandas as pd
import warnings
from datetime import date
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from models import FeedForward, LSTM, Hybrid
import torch.backends.cudnn as cudnn
torch.backends.cuda.matmul.allow_tf32 = True
cudnn.allow_tf32 = True
cudnn.benchmark = True

warnings.filterwarnings('ignore')

In [2]:
# ...existing code...
# Add after your imports (or near training utils)
import os
from torch.utils.data import Dataset, IterableDataset

def compute_valid_indices(tickers: np.ndarray, seq_len: int) -> np.ndarray:
    """Indices i where window [i-seq_len:i) stays within same ticker segment."""
    t = np.asarray(tickers)
    n = len(t)
    idx = []
    start = 0
    for i in range(1, n + 1):
        if i == n or t[i] != t[i - 1]:
            end = i
            if end - start > seq_len:
                idx.extend(range(start + seq_len, end))
            start = end
    return np.asarray(idx, dtype=np.int64)

class SequenceDataset(Dataset):
    """Lazily yields sliding windows without materializing X_seq in RAM."""
    def __init__(self, X: np.ndarray, y: np.ndarray, tickers: np.ndarray, seq_len: int, to_dtype=torch.float32):
        self.X = torch.from_numpy(X.astype(np.float32, copy=False))
        self.y = torch.from_numpy(y.astype(np.float32, copy=False))
        self.seq_len = seq_len
        self.valid_idx = compute_valid_indices(tickers, seq_len)
        self.to_dtype = to_dtype

    def __len__(self):
        return len(self.valid_idx)

    def __getitem__(self, i):
        idx = self.valid_idx[i]
        x = self.X[idx - self.seq_len: idx].to(dtype=self.to_dtype).contiguous()  # CPU tensor
        y = self.y[idx].to(dtype=self.to_dtype)
        return x, y
    
class CUDAPrefetchLoader:
    def __init__(self, loader, device, dtype, prefetch=4):
        self.loader, self.device, self.dtype, self.prefetch = loader, device, dtype, prefetch
        self.stream = torch.cuda.Stream() if device.type == 'cuda' else None

    def __iter__(self):
        if self.stream is None:
            yield from self.loader
            return

        it, cache = iter(self.loader), []
        with torch.cuda.stream(self.stream):
            for _ in range(self.prefetch):
                try:
                    bx, by = next(it)
                except StopIteration:
                    break
                cache.append((
                    bx.to(self.device, dtype=self.dtype, non_blocking=True),
                    by.to(self.device, dtype=self.dtype, non_blocking=True)
                ))

        while cache:
            torch.cuda.current_stream().wait_stream(self.stream)
            batch = cache.pop(0)

            try:
                nx, ny = next(it)
                with torch.cuda.stream(self.stream):
                    cache.append((
                        nx.to(self.device, dtype=self.dtype, non_blocking=True),
                        ny.to(self.device, dtype=self.dtype, non_blocking=True)
                    ))
            except StopIteration:
                pass

            yield batch
    def __len__(self):
        return len(self.loader)

In [3]:
# READ S&P 500 TICKERS FROM TXT
def load_simple_ticker_list(filename='sp500_tickers.txt'):
    """Load tickers from simple text file (one per line)"""
    try:
        with open(filename, 'r') as f:
            tickers = [line.strip() for line in f.readlines() if line.strip()]
        return tickers
    except FileNotFoundError:
        print(f"❌ File {filename} not found.")
        return []

tickers = load_simple_ticker_list('sp500_tickers.txt')

In [ ]:
# S&P 500 STOCK PROCESSING WITH LIMITS
import gc  # For garbage collection

# Configuration
sp500_tickers = tickers  # From loaded file

# STOCK LIMIT CONFIGURATION
USE_FULL_DATASET = True  # Set to True when ready for full run
STOCK_LIMIT = 5          # Limit for testing (set to desired number)

# Apply stock limit
if USE_FULL_DATASET:
    processing_tickers = sp500_tickers
    print(f"🚀 FULL DATASET MODE: Processing all {len(processing_tickers)} S&P 500 stocks")
else:
    processing_tickers = sp500_tickers[:STOCK_LIMIT]
    print(f"🧪 TEST MODE: Processing first {len(processing_tickers)} stocks")

print("=" * 60)
print(f"📊 Dataset: S&P 500")
print(f"🎯 Stocks to process: {len(processing_tickers)}")
print("=" * 60)

all_data = []
total_tickers = len(processing_tickers)

# For progress tracking
successful_stocks = 0
failed_stocks = 0

for idx, ticker in enumerate(processing_tickers, 1):
    try:
        features = pd.read_parquet(f'processed_features_cleaned/{ticker}.parquet')
        
        # Add target and ticker
        features['Target'] = features['Close'].shift(-1) / features['Close'] - 1.0
        features['Ticker'] = ticker
        
        # Remove infinite and NaN values
        clean_data = features.dropna()
        
        all_data.append(clean_data)
        
        # GARBAGE COLLECTION
        if idx % 25 == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print(f"\r🧹 [{idx}/{total_tickers}] Memory cleanup completed" + " " * 30)
            
    except Exception as e:
        print(f"\r❌ [{idx}/{total_tickers}] {ticker}: Error - {str(e)}")
        continue

# Final cleanup
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Combine all data
if all_data:
    ml_data = pd.concat(all_data, ignore_index=True)
    feature_count = len([col for col in ml_data.columns if col not in ['Target', 'Ticker']])
    
    print(f"📊 Total samples: {len(ml_data):,}")
    print(f"🔧 Features: {feature_count}")
    print(f"📈 Stocks: {len(ml_data['Ticker'].unique())}")
    print(f"💾 Memory: {ml_data.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
    
    # Show sample data
    sample_cols = ['Price_Change_1d', 'RSI_15m', 'MACD_15m', 'ATR_14_15m', 'Target', 'Ticker']
    print(f"\n📋 SAMPLE DATA:")
    print(ml_data[sample_cols].head(3).round(4))    
else:
    print("\n❌ ERROR: No data processed successfully!")
    ml_data = pd.DataFrame()
    
del all_data
gc.collect()

In [ ]:
#Device Detection

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print("Using Apple Silicon GPU")
else:
    device = torch.device('cpu')
    print("Using CPU")

In [ ]:
# DATA PREPARATION FOR PYTORCH
print("📊 PREPARING DATA FOR PYTORCH")
print("=" * 40)

from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from typing import Any

def prep_training_data(ml_data : pd.DataFrame, sequence_length=10, test_size=0.2, scale_target=True, log_transform_target=True) -> dict[str, Any]:
    """
    Prepare data for PyTorch training
    
    Args:
        ml_data (pd.DataFrame): Your dataset
        sequence_length (int): Number of days to look back (for LSTM)
        test_size (int): Fraction for testing
        scale_target (bool): Whether to scale the target variable
        log_transform_target (bool): Whether to transform data in log format
        
    Returns:
        data (dict): With torch float sensors sent to the {device}
    """
    print(f"📋 Input data shape: {ml_data.shape}")
    
    # Separate features and target
    feature_cols = [col for col in ml_data.columns if col not in ['Target', 'Ticker']]    
    ml_data.dropna(inplace=True)

    
    if log_transform_target:
            print("Applying log1p transformation to the target variable.")
            ml_data['Target'] = np.log1p(ml_data['Target']) # log1p is log(1+x), safer for values near zero

    # --- Split data using train_test_split WITHOUT shuffling ---
    X_train_df, X_test_df, y_train_df, y_test_df, t_train, t_test = train_test_split(
        ml_data[feature_cols], 
        ml_data['Target'],
        ml_data['Ticker'],
        test_size=test_size,
        shuffle=False, # CRUCIAL: Do not shuffle time-series data
        random_state=42 # Ensures reproducibility, though not strictly needed with shuffle=False
    )
    
    del ml_data
    gc.collect()

    # --- FIX: Fit scaler ONLY on training data ---
    feature_scaler = RobustScaler()
    X_train = feature_scaler.fit_transform(X_train_df).astype(np.float32)
    X_test = feature_scaler.transform(X_test_df).astype(np.float32) # Use transform, not fit_transform
    
    # Handle target scaling correctly
    y_train = y_train_df.values.astype(np.float32)
    y_test = y_test_df.values.astype(np.float32)
    t_train = np.array(t_train)
    t_test = np.array(t_test)
    
    target_scaler = None
    if scale_target:
        target_scaler = RobustScaler()
        y_train = target_scaler.fit_transform(y_train.reshape(-1, 1)).flatten()
        y_test = target_scaler.transform(y_test.reshape(-1, 1)).flatten()
    
    print(f"💾 Memory after FP32: {(X_train.nbytes + X_test.nbytes + y_train.nbytes + y_test.nbytes) / 1024**2:.1f} MB")
    
    # Create sequences that do NOT cross ticker boundaries
    def create_sequences(X, y, tickers, seq_len):
        X_seq, y_seq = [], []
        for i in range(seq_len, len(X)):
            if np.all(tickers[i-seq_len:i] == tickers[i]):  # same ticker across the window
                X_seq.append(X[i-seq_len:i])
                y_seq.append(y[i])
        return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.float32)

    # X_train_seq, y_train_seq = create_sequences(X_train, y_train, t_train, sequence_length)
    # X_test_seq, y_test_seq = create_sequences(X_test, y_test, t_test, sequence_length)
    X_train_seq = y_train_seq = X_test_seq = y_test_seq = None

    # Convert to PyTorch tensors
    data = {
        'regular': {
            'X_train': torch.from_numpy(X_train),  # CPU float32
            'X_test': torch.from_numpy(X_test),
            'y_train': torch.from_numpy(y_train),
            'y_test': torch.from_numpy(y_test)
        },
        'sequence': {
            'X_train': None,  # will be built lazily by Dataset
            'X_test': None,
            'y_train': None,
            'y_test': None
        },
        'raw': {  # needed to build SequenceDataset lazily
            'X_train': X_train, 'y_train': y_train, 't_train': t_train,
            'X_test': X_test,   'y_test': y_test,   't_test': t_test
        },
        'scalers': {
            'feature_scaler': feature_scaler,
            'target_scaler': target_scaler
        },
        'log_transform_target': log_transform_target,
        'feature_names': feature_cols,
        "predicting_returns": True
    }
    
    print(f"✅ Regular data - Train: {X_train.shape}, Test: {X_test.shape}")
    print(f"✅ Sequence data - built lazily in DataLoader")
    print(f"📊 Features: {len(feature_cols)}")
    print(f"🎯 Target scaling: {'Yes' if scale_target else 'No'}")
    
    del X_train, X_test, y_train, y_test, X_train_seq, y_train_seq, X_test_seq, y_test_seq
    gc.collect()
    
    return data

training_data = prep_training_data(ml_data,
                                    sequence_length=8,
                                    test_size=0.1,
                                    scale_target=False,
                                    log_transform_target=False
                                    )

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("=" * 40)

In [ ]:

# PYTORCH TRAINING FUNCTIONS AND EXECUTION
from torch import autocast, GradScaler

def train_pytorch_model(model, data, epochs, model_type, batch_size, lr, use_mixed_precision=False):
    """
    Train PyTorch model and track directional accuracy (Supports GPU)
    
    Args:
        model: PyTorch model
        data: Prepared data dictionary  
        model_type: 'regular' or 'sequence'
        epochs: Number of training epochs
        batch_size: Batch size for training
        lr: Learning rate
        use_mixed_precision: Use mixed precision, will automatically be set to true if cuda is detected
    """ 
    # --- BF16 detection ---
    supports_bf16 = (device.type == 'cuda' and torch.cuda.is_bf16_supported())
    sequence_mode = (model_type == 'sequence')
    if sequence_mode:
        # Force FP32 path for cuDNN LSTM stability
        supports_bf16 = False
        use_mixed_precision = False

    # Move model and set dtype
    if supports_bf16:
        model = model.to(device).to(dtype=torch.bfloat16)
        use_mixed_precision = False  # Disable GradScaler for BF16
    elif device.type == 'cuda':
        model = model.to(device)     # Keep FP32 params (use FP16 AMP if you want)
        use_mixed_precision = (not sequence_mode)
    else:
        model = model.to(device)
        use_mixed_precision = False
        
    # torch.compile can be flaky with cuDNN LSTM; skip for sequence models
    if not sequence_mode:
        try:
            model = torch.compile(model, mode="default", dynamic=True)
        except Exception as e:
            print("compile skipped:", e)
    
    # Build datasets on CPU
    if model_type == 'sequence':
        seq_len = 16 if 'sequence_length' not in globals() else 16  # adjust if needed
        Xtr = data['raw']['X_train']; ytr = data['raw']['y_train']; ttr = data['raw']['t_train']
        Xte = data['raw']['X_test'];  yte = data['raw']['y_test'];  tte = data['raw']['t_test']

        train_dataset = SequenceDataset(Xtr, ytr, ttr, seq_len, to_dtype=torch.float32)
        test_dataset  = SequenceDataset(Xte, yte, tte, seq_len, to_dtype=torch.float32)
    else:
        Xtr = data['regular']['X_train'].float()  # CPU tensors
        ytr = data['regular']['y_train'].float()
        Xte = data['regular']['X_test'].float()
        yte = data['regular']['y_test'].float()

        train_dataset = TensorDataset(Xtr, ytr)
        test_dataset  = TensorDataset(Xte, yte)
    
    # DataLoaders: pin memory + workers to stream batches
    num_workers = max(1, os.cpu_count() // 2)
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=(model_type!='sequence'),
        drop_last=True, pin_memory=(device.type=='cuda'), num_workers=num_workers,
        persistent_workers=(num_workers>0), prefetch_factor=2
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False,
        pin_memory=(device.type=='cuda'), num_workers=num_workers,
        persistent_workers=(num_workers>0), prefetch_factor=2
    )
    
    steps_per_epoch = len(train_loader)
    
    if device.type == 'cuda':
        batch_dtype = torch.bfloat16 if supports_bf16 else torch.float32
        if model_type == 'sequence':
            batch_dtype = torch.float32  # keep LSTM inputs FP32 for cuDNN stability
        train_loader = CUDAPrefetchLoader(train_loader, device, batch_dtype, prefetch=4)
        test_loader  = CUDAPrefetchLoader(test_loader,  device, batch_dtype, prefetch=2)


    # Loss function and optimizer
    # In your train_pytorch_model function, replace the criterion with:
    class DirectionalLoss(nn.Module):
        def __init__(self, alpha=0.1, margin=0.0):
            super().__init__()
            self.alpha = alpha
            self.margin = margin
            self.mse = nn.MSELoss()
            
        def forward(self, pred, target):
            # MSE component
            mse_loss = self.mse(pred, target)
            
            # Directional component (penalize wrong direction)
            dir_penalty = torch.relu(self.margin - (pred * target)).mean()
            return (1 - self.alpha) * mse_loss + self.alpha * dir_penalty

    criterion = DirectionalLoss(alpha=0.05, margin=0.0)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    # scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.1)
    
## Consider using OneCycleLR
    warmup_epochs = 5
    total_steps = len(train_loader) * epochs
    warmup_steps = len(train_loader) * warmup_epochs

    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, 
        max_lr=lr,
        epochs=epochs,
        # total_steps=total_steps,
        steps_per_epoch=steps_per_epoch,
        pct_start=min(0.99, warmup_epochs / max(1, epochs))
    )
    
    scaler = GradScaler(enabled=(device.type == 'cuda' and use_mixed_precision and not supports_bf16))
    
    # Training metrics
    train_losses, test_losses, directional_accuracies = [], [], []
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for batch_X, batch_y in train_loader:
            # Move per-batch to GPU with BF16 (saves VRAM)
            if device.type != 'cuda':
                batch_X = batch_X.to(device)
                batch_y = batch_y.to(device)
            batch_X = batch_X.contiguous()
            optimizer.zero_grad(set_to_none=True)
            
            if supports_bf16:
                # BF16: autocast + regular backward (no GradScaler)
                with autocast(device_type='cuda', dtype=torch.bfloat16):
                    outputs = model(batch_X).squeeze()
                    loss = criterion(outputs, batch_y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            elif use_mixed_precision and device.type == 'cuda': # RTX 4060
                # Optional FP16 path (if you ever use it)
                with autocast(device_type='cuda', dtype=torch.float16):
                    outputs = model(batch_X).squeeze()
                    loss = criterion(outputs, batch_y)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()

            else: # MPS
                # Pure FP32
                outputs = model(batch_X).squeeze()
                loss = criterion(outputs, batch_y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            scheduler.step()
            train_loss += float(loss.detach().cpu())
            
        # Eval
        model.eval()
        test_loss = 0.0
        epoch_predictions, epoch_actuals = [], []
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X = batch_X.contiguous()
                if device.type == 'cuda' and supports_bf16:
                    batch_X = batch_X.to(device, dtype=torch.bfloat16, non_blocking=True)
                    batch_y = batch_y.to(device, dtype=torch.bfloat16, non_blocking=True)
                    with autocast(device_type='cuda', dtype=torch.bfloat16):
                        outputs = model(batch_X).squeeze()
                        loss = criterion(outputs, batch_y)
                elif device.type == 'cuda' and use_mixed_precision:
                    batch_X = batch_X.to(device, non_blocking=True)
                    batch_y = batch_y.to(device, non_blocking=True)
                    with autocast(device_type='cuda', dtype=torch.float16):
                        outputs = model(batch_X).squeeze()
                        loss = criterion(outputs, batch_y)
                else:
                    batch_X = batch_X.to(device)
                    batch_y = batch_y.to(device)
                    outputs = model(batch_X).squeeze()
                    loss = criterion(outputs, batch_y)

                test_loss += float(loss.detach().cpu())
                epoch_predictions.extend(outputs.float().cpu().numpy())
                epoch_actuals.extend(batch_y.float().cpu().numpy())

        
        # Calculate directional accuracy for this epoch
        epoch_preds = np.array(epoch_predictions)
        epoch_acts = np.array(epoch_actuals)
        
        epoch_preds_reverted = epoch_preds
        epoch_acts_reverted = epoch_acts
        
        # Calculate directional accuracy
        direction_correct = np.sign(epoch_acts_reverted) == np.sign(epoch_preds_reverted)
        epoch_accuracy = np.mean(direction_correct) * 100
        
        # Calculate average losses
        avg_train_loss = train_loss / len(train_loader)
        avg_test_loss = test_loss / len(test_loader)
        
        train_losses.append(avg_train_loss)
        test_losses.append(avg_test_loss)
        directional_accuracies.append(epoch_accuracy)
        
        print(f'Epoch {epoch:3d}: Train Loss: {avg_train_loss:.6f}, Test Loss: {avg_test_loss:.6f}, Dir Acc: {epoch_accuracy:.1f}%')

    # Final evaluation
    model.eval()
    predictions, actuals = [], []
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X = batch_X.contiguous()
            if device.type == 'cuda' and supports_bf16:
                batch_X = batch_X.to(device, dtype=torch.bfloat16, non_blocking=True)
                batch_y = batch_y.to(device, dtype=torch.bfloat16, non_blocking=True)
                with autocast(device_type='cuda', dtype=torch.bfloat16):
                    outputs = model(batch_X).squeeze()
            elif device.type == 'cuda' and use_mixed_precision:
                batch_X = batch_X.to(device, non_blocking=True)
                batch_y = batch_y.to(device, non_blocking=True)
                with autocast(device_type='cuda', dtype=torch.float16):
                    outputs = model(batch_X).squeeze()
            else:
                batch_X = batch_X.to(device)
                batch_y = batch_y.to(device)
                outputs = model(batch_X).squeeze()

            predictions.extend(outputs.float().cpu().numpy().flatten().tolist())
            actuals.extend(batch_y.float().cpu().numpy().flatten().tolist())

    # --- NEW: REVERT TRANSFORMATIONS BEFORE CALCULATING METRICS ---
    predictions_np = np.array(predictions).reshape(-1, 1)
    actuals_np = np.array(actuals).reshape(-1, 1)
        
    # Invert scaling if used
    tgt_scaler = data['scalers'].get('target_scaler', None)
    if tgt_scaler is not None:
        predictions_np = tgt_scaler.inverse_transform(predictions_np)
        actuals_np = tgt_scaler.inverse_transform(actuals_np)
    # Invert log transform if used
    if data.get('log_transform_target', False):
        predictions_np = np.expm1(predictions_np)
        actuals_np = np.expm1(actuals_np)
    
    # Additional safety: clip final results to reasonable return range
    predictions_np = np.clip(predictions_np, -0.5, 0.5)  # -50% to +50% returns
    actuals_np = np.clip(actuals_np, -0.5, 0.5)
    
    # Calculate metrics
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    
    mse = mean_squared_error(actuals_np, predictions_np)
    mae = mean_absolute_error(actuals_np, predictions_np)
    r2 = r2_score(actuals_np, predictions_np)
    
    print(f"📊 Final Metrics - MSE: {mse:.6f}, MAE: {mae:.6f}, R²: {r2:.4f}")
    
    if USE_FULL_DATASET:
            torch.save(model.state_dict(), f'models_full_dataset/{model.__class__.__name__.lower()}_f.pth')
    else:
        torch.save(model.state_dict(), f'models_limited/{model.__class__.__name__.lower()}_l.pth')
    
    return {
        'model': model,
        'train_losses': train_losses,
        'test_losses': test_losses,
        'directional_accuracies': directional_accuracies,
        'predictions': predictions_np.flatten().tolist(), # Return reverted predictions
        'actuals': actuals_np.flatten().tolist(),       # Return reverted actuals
        'metrics': {'mse': mse, 'mae': mae, 'r2': r2}
    }

In [ ]:
# TRAIN ALL MODELS
print(f"🔥 TRAINING NEURAL NETWORKS ON {device}")
print("=" * 50)

if USE_FULL_DATASET:
    global_epochs = 8
    global_batch_size = 256     # Larger batches for efficiency
    global_lr = 0.005          # Lower learning rate for stability
else:
    global_epochs = 20
    global_batch_size = 32
    global_lr = 0.02
    
results = {}

# 2. Train LSTM Network
print("\n2️⃣ TRAINING LSTM NETWORK")
print("-" * 25)
lstm_model = LSTM(input_features=len(training_data['feature_names'])).to(device)
results['LSTM'] = train_pytorch_model(
    lstm_model, training_data, model_type='sequence', epochs=global_epochs,
    batch_size=512,
    lr=global_lr
)

# 1. Train Feed-Forward Neural Network
print("\n1️⃣ TRAINING FEED-FORWARD NEURAL NETWORK")
print("-" * 45)
ff_model = FeedForward(input_features=len(training_data['feature_names'])).to(device)
results['FeedForward'] = train_pytorch_model(
    ff_model, training_data, model_type='regular', epochs=global_epochs,
    batch_size=4096,
    lr=global_lr
)


# 3. Train Hybrid Network
print("\n3️⃣ TRAINING HYBRID CNN-LSTM NETWORK") 
print("-" * 35)
hybrid_model = Hybrid(input_features=len(training_data['feature_names'])).to(device)
results['Hybrid'] = train_pytorch_model(
    hybrid_model, training_data, model_type='sequence', epochs=global_epochs,
    batch_size=512,
    lr=global_lr
)
print("\n✅ PyTorch GPU Training Complete!")
print("=" * 50)

In [ ]:
## HELPER METHODS

# Calculate enhanced metrics for all models
def calculate_enhanced_metrics(results, training_data):
    """Calculate trading-focused metrics for model evaluation"""
    enhanced_results = {}
    
    log_transform_used = training_data['log_transform_target']
    outlier_threshold = 10 if log_transform_used else 50
    
    for name, result in results.items():
        actuals = np.array(result['actuals'])
        predictions = np.array(result['predictions'])
        
        # Convert to percentage returns
        actual_returns = actuals * 100
        predicted_returns = predictions * 100
        
        # Filter outliers
        mask = (np.abs(actual_returns) < outlier_threshold) & (np.abs(predicted_returns) < outlier_threshold)
        actual_clean = actual_returns[mask]
        predicted_clean = predicted_returns[mask]

        # Trading-focused metrics
        direction_correct = np.sign(actual_clean) == np.sign(predicted_clean)
        directional_accuracy = np.mean(direction_correct) * 100
        
        # Hit rate for significant moves (>1% moves)
        significant_moves = np.abs(actual_clean) > 1.0
        if np.sum(significant_moves) > 0:
            hit_rate_significant = np.mean(direction_correct[significant_moves]) * 100
        else:
            hit_rate_significant = 0
        
        # Return prediction accuracy
        return_mae = np.mean(np.abs(actual_clean - predicted_clean))
        return_rmse = np.sqrt(np.mean((actual_clean - predicted_clean)**2))
        
        # Volatility matching
        actual_vol = np.std(actual_clean)
        predicted_vol = np.std(predicted_clean)
        vol_ratio = predicted_vol / actual_vol if actual_vol > 0 else 0
        
        # Sharpe-like ratio for predictions (mean return / volatility)
        mean_predicted_return = np.mean(predicted_clean)
        sharpe_like = mean_predicted_return / predicted_vol if predicted_vol > 0 else 0
        
        # Correlation
        correlation = np.corrcoef(actual_clean, predicted_clean)[0, 1]
        
        enhanced_results[name] = {
            'directional_accuracy': directional_accuracy,
            'hit_rate_significant': hit_rate_significant,
            'return_mae': return_mae,
            'return_rmse': return_rmse,
            'correlation': correlation,
            'actual_vol': actual_vol,
            'predicted_vol': predicted_vol,
            'vol_ratio': vol_ratio,
            'sharpe_like': sharpe_like,
            'actual_clean': actual_clean,
            'predicted_clean': predicted_clean,
            'samples': len(actual_clean)
        }
    
    return enhanced_results

# Find best model based on composite score
def calculate_composite_score(metrics):
    """Calculate composite score emphasizing trading performance"""
    # Normalize metrics (higher is better)
    dir_acc_norm = metrics['directional_accuracy'] / 100  # 0-1 scale
    corr_norm = (metrics['correlation'] + 1) / 2  # -1,1 to 0,1 scale
    vol_ratio_norm = 1 - abs(1 - metrics['vol_ratio'])  # Penalty for being far from 1
    
    # Penalty for high MAE (lower is better)
    mae_penalty = 1 / (1 + metrics['return_mae'])  # Higher MAE = lower score
    
    # Composite score (weights can be adjusted)
    composite = (
        0.4 * dir_acc_norm +      # 40% directional accuracy
        0.3 * corr_norm +         # 30% correlation
        0.2 * mae_penalty +       # 20% MAE penalty
        0.1 * vol_ratio_norm      # 10% volatility matching
    )
    
    return composite

In [ ]:
# IMPROVED PYTORCH RESULTS VISUALIZATION - TRUE BLACK DARK MODE

import matplotlib.pyplot as plt
import matplotlib

# Set dark theme for plots with TRUE BLACK backgrounds
plt.style.use('dark_background')
matplotlib.rcParams.update({
    'figure.facecolor': '#000000',      # Pure black
    'axes.facecolor': '#000000',        # Pure black
    'savefig.facecolor': '#000000',     # Pure black
    'text.color': 'white',
    'axes.labelcolor': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'axes.edgecolor': 'white',
    'grid.color': '#404040',
    'axes.spines.bottom': True,
    'axes.spines.top': True,
    'axes.spines.right': True,
    'axes.spines.left': True
})

# Bright colors that pop against pure black
dark_colors = {
    'blue': '#00E5FF', 'purple': '#C77DFF', 'orange': '#FF9F40',
    'red': '#FF5555', 'coral': '#FF8A80', 'teal': '#1DE9B6',
    'yellow': '#FFEB3B', 'green': '#69F0AE'
}

# Define models dictionary and calculate metrics
models = {name: result['model'] for name, result in results.items()}
enhanced_metrics = calculate_enhanced_metrics(results, training_data)
composite_scores = {name: calculate_composite_score(metrics) for name, metrics in enhanced_metrics.items()}
best_model_enhanced = max(composite_scores, key=lambda k: composite_scores[k])

# Create visualization with accuracy plot
fig = plt.figure(figsize=(30, 22), facecolor='#000000')
gs = fig.add_gridspec(5, 4, hspace=0.35, wspace=0.3)

fig.suptitle(f'Performance Analysis: {global_epochs} epochs, {STOCK_LIMIT} stocks, {len(training_data["feature_names"])} params, {date.today()}', 
             fontsize=16, fontweight='bold', color='white')
fig.text(0.5, 0.96, f"Log transform: {training_data['log_transform_target']}", ha='center', fontsize=13, color='white')

models_list = list(enhanced_metrics.keys())

def create_bar_chart(ax, x_data, y_data_list, labels, colors, title, ylabel, loc, add_random_line=False):
    """Helper function to create consistent bar charts"""
    ax.set_facecolor('#000000')
    x = np.arange(len(x_data))
    width = 0.25 if len(y_data_list) == 3 else 0.4
    
    bars_list = []
    for i, (y_data, label, color) in enumerate(zip(y_data_list, labels, colors)):
        offset = (i - len(y_data_list)//2) * width
        bars = ax.bar(x + offset, y_data, width, label=label, alpha=0.9, color=color)
        bars_list.append(bars)
        
        # Add value labels
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + (0.05 if height < 10 else 1),
                    f'{height:.1f}' if height > 1 else f'{height:.2f}', 
                    ha='center', va='bottom', fontsize=9, color='white', fontweight='bold')
    
    if add_random_line:
        ax.axhline(y=50, color='#FF5555', linestyle='--', alpha=0.8, linewidth=2)
    
    ax.set_title(title, fontweight='bold', fontsize=14, color='white')
    ax.set_xlabel('Models', color='white')
    ax.set_ylabel(ylabel, color='white')
    ax.set_ylim(0, np.max(y_data_list) + 0.25 * np.max(y_data_list))
    ax.set_xticks(x)
    ax.set_xticklabels(x_data, color='white')
    ax.legend(framealpha=0.9, facecolor='#1a1a1a', edgecolor='white', loc=loc)
    ax.grid(True, alpha=0.3, color='#505050')
    
    return bars_list

# 1. Trading Performance Dashboard (Top row, spans 2 columns)
ax1 = fig.add_subplot(gs[0, :2])
dir_acc = [enhanced_metrics[m]['directional_accuracy'] for m in models_list]
hit_rate_sig = [enhanced_metrics[m]['hit_rate_significant'] for m in models_list]
correlation = [enhanced_metrics[m]['correlation'] * 100 for m in models_list]

create_bar_chart(ax1, models_list, 
                [dir_acc, hit_rate_sig, correlation],
                ['Directional Accuracy (%)', 'Hit Rate >1% moves (%)', 'Correlation (%)'],
                [dark_colors['blue'], dark_colors['purple'], dark_colors['orange']],
                'Trading Performance Metrics', 'Performance (%)', 'lower left', add_random_line=True)

# 2. Return Prediction Quality (Top row, right)
ax2 = fig.add_subplot(gs[0, 2:])
mae_values = [enhanced_metrics[m]['return_mae'] for m in models_list]
rmse_values = [enhanced_metrics[m]['return_rmse'] for m in models_list]

create_bar_chart(ax2, models_list,
                [mae_values, rmse_values],
                ['MAE (pp)', 'RMSE (pp)'],
                [dark_colors['red'], dark_colors['coral']],
                'Return Prediction Errors', 'Error (Percentage Points)', 'lower left')

# 3. NEW: Directional Accuracy Over Epochs (Second row, spans full width)
ax3 = fig.add_subplot(gs[1, :])
ax3.set_facecolor('#000000')

colors_list = [dark_colors['blue'], dark_colors['purple'], dark_colors['orange']]
for i, (model_name, result) in enumerate(results.items()):
    if 'directional_accuracies' in result:
        epochs = range(len(result['directional_accuracies']))
        accuracies = result['directional_accuracies']
        
        ax3.plot(epochs, accuracies, label=f'{model_name}', 
                color=colors_list[i % len(colors_list)], linewidth=3, alpha=0.9, marker='o', markersize=4)
        
        # Add final accuracy value as text
        final_acc = accuracies[-1]
        ax3.text(len(epochs)-1, final_acc, f'{final_acc:.1f}%', 
                color=colors_list[i % len(colors_list)], fontweight='bold', fontsize=10,
                ha='left', va='center')

ax3.set_title('Directional Accuracy Over Training Epochs', fontweight='bold', fontsize=16, color='white')
ax3.set_xlabel('Epoch', color='white', fontsize=12)
ax3.set_ylabel('Directional Accuracy (%)', color='white', fontsize=12)
ax3.legend(framealpha=0.9, facecolor='#1a1a1a', edgecolor='white', fontsize=12)
ax3.grid(True, alpha=0.3, color='#505050')
ax3.axhline(y=50, color='#FF5555', linestyle='--', alpha=0.8, linewidth=2, label='Random Baseline (50%)')
ax3.axhline(y=60, color='#00FF00', linestyle=':', alpha=0.6, linewidth=1, label='Good Performance (60%)')

# Set y-axis limits for better visualization
if any('directional_accuracies' in result for result in results.values()):
    max_acc = max([max(result['directional_accuracies']) for result in results.values() if 'directional_accuracies' in result])
    ax3.set_ylim(45, max_acc + 5)

# 4. Best Model Detailed Analysis (Third row, left)
ax4 = fig.add_subplot(gs[2, :2])
ax4.set_facecolor('#000000')
if best_model_enhanced in enhanced_metrics:
    best_metrics = enhanced_metrics[best_model_enhanced]
    actual_clean = best_metrics['actual_clean']
    predicted_clean = best_metrics['predicted_clean']
    
    # Color code: green for correct direction, red for incorrect
    colors = [dark_colors['green'] if np.sign(a) == np.sign(p) else dark_colors['red'] 
              for a, p in zip(actual_clean, predicted_clean)]
    
    ax4.scatter(actual_clean, predicted_clean, c=colors, alpha=0.8, s=30, edgecolors='white', linewidth=0.7)
    
    # Perfect prediction line
    min_val, max_val = min(min(actual_clean), min(predicted_clean)), max(max(actual_clean), max(predicted_clean))
    ax4.plot([min_val, max_val], [min_val, max_val], color='white', linestyle='--', alpha=0.9, linewidth=3)
    
    ax4.set_title(f'{best_model_enhanced}: Actual vs Predicted Returns', fontweight='bold', fontsize=14, color='white')
    ax4.set_xlabel('Actual Returns (%)', color='white')
    ax4.set_ylabel('Predicted Returns (%)', color='white')
    ax4.grid(True, alpha=0.3, color='#505050')
    ax4.axhline(y=0, color='#808080', linestyle='-', alpha=0.6)
    ax4.axvline(x=0, color='#808080', linestyle='-', alpha=0.6)

# 5. Volatility Analysis (Third row, right)
ax5 = fig.add_subplot(gs[2, 2:])
actual_vols = [enhanced_metrics[m]['actual_vol'] for m in models_list]
predicted_vols = [enhanced_metrics[m]['predicted_vol'] for m in models_list]

create_bar_chart(ax5, models_list,
                [actual_vols, predicted_vols],
                ['Actual Volatility', 'Predicted Volatility'],
                [dark_colors['teal'], dark_colors['yellow']],
                'Volatility Matching', 'Volatility (%)', 'lower right')

# Add ratio labels for volatility
for i, (actual, predicted) in enumerate(zip(actual_vols, predicted_vols)):
    ratio = predicted / actual if actual > 0 else 0
    ax5.text(i, max(actual, predicted) + 0.1, f'Ratio: {ratio:.2f}', 
             ha='center', va='bottom', fontsize=9, fontweight='bold', color='white')

# 6. Model Comparison Radar Chart (Fourth row, left)
ax6 = fig.add_subplot(gs[3, :2], projection='polar')
ax6.set_facecolor('#000000')

metrics_names = ['Directional\nAccuracy', 'Correlation', 'Low MAE', 'Vol Match', 'Composite\nScore']
angles = np.linspace(0, 2 * np.pi, len(metrics_names), endpoint=False).tolist() + [0]

for i, model in enumerate(models_list):
    if model in enhanced_metrics:
        m = enhanced_metrics[model]
        values = [
            m['directional_accuracy'] / 100,
            (m['correlation'] + 1) / 2,
            1 / (1 + m['return_mae']),
            1 - abs(1 - m['vol_ratio']),
            composite_scores[model]
        ] + [m['directional_accuracy'] / 100]  # Close the circle
        
        ax6.plot(angles, values, 'o-', linewidth=4, label=model, alpha=0.9, color=colors_list[i % len(colors_list)])
        ax6.fill(angles, values, alpha=0.2, color=colors_list[i % len(colors_list)])

ax6.set_xticks(angles[:-1])
ax6.set_xticklabels(metrics_names, color='white')
ax6.set_ylim(0, 1)
ax6.set_title('Multi-Metric Model Comparison', fontweight='bold', fontsize=14, pad=20, color='white')
ax6.legend(loc='upper left', bbox_to_anchor=(1.4, 1.0), framealpha=0.9, facecolor='#1a1a1a', edgecolor='white')
ax6.grid(True, color='#505050')
ax6.tick_params(colors='white')

# 7. Training Loss Evolution (Fourth row, right)
ax7 = fig.add_subplot(gs[3, 2])
ax7.set_facecolor('#000000')

for i, (name, result) in enumerate(results.items()):
    epochs_range = range(len(result['train_losses']))
    ax7.plot(epochs_range, result['train_losses'], label=f'{name} Train', alpha=0.9, linewidth=3, color=colors_list[i % len(colors_list)])

ax7.set_title('Loss Convergence Analysis', fontweight='bold', fontsize=14, color='white')
ax7.set_xlabel('Epoch', color='white')
ax7.set_ylabel('Loss', color='white')
ax7.legend(framealpha=0.9, facecolor='#1a1a1a', edgecolor='white')
ax7.grid(True, alpha=0.3, color='#505050')
ax7.set_yscale('log')

# 8. Validation Loss Evolution (Fourth row, far right)
ax8 = fig.add_subplot(gs[3, 3])
ax8.set_facecolor('#000000')

# Plot only validation loss
for i, (name, result) in enumerate(results.items()):
    epochs_range = range(len(result['test_losses']))
    ax8.plot(epochs_range, result['test_losses'], label=f'{name} Val', linestyle='--', alpha=0.9, linewidth=3, color=colors_list[i % len(colors_list)])

ax8.set_title('Validation Loss Convergence', fontweight='bold', fontsize=14, color='white')
ax8.set_xlabel('Epoch', color='white')
ax8.set_ylabel('Validation Loss', color='white')
ax8.legend(framealpha=0.9, facecolor='#1a1a1a', edgecolor='white')
ax8.grid(True, alpha=0.3, color='#505050')
ax8.set_yscale('log')

# 8. Performance Summary Table (Bottom row, spans full width)
ax9 = fig.add_subplot(gs[4, :])
ax9.axis('tight')
ax9.axis('off')
ax9.set_facecolor('#000000')

headers = ['Model', 'Dir. Acc.', 'Hit Rate >1%', 'Actual range', 'Predicted range', 'MAE (pp)', 'Correlation', 'Vol Ratio', 'Composite Score', 'Samples']
table_data = []

for model in models_list:
    if model in enhanced_metrics:
        m = enhanced_metrics[model]
        table_data.append([
            model,
            f"{m['directional_accuracy']:.1f}%",
            f"{m['hit_rate_significant']:.1f}%",
            f"{np.min(m['actual_clean']):.1f}% - {np.max(m['actual_clean']):.1f}%",
            f"{np.min(m['predicted_clean']):.1f}% - {np.max(m['predicted_clean']):.1f}%",
            f"{m['return_mae']:.2f}",
            f"{m['correlation']:.3f}",
            f"{m['vol_ratio']:.2f}",
            f"{composite_scores[model]:.3f}",
            f"{m['samples']:,}"
        ])

table = ax9.table(cellText=table_data, colLabels=headers, cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 3)

# Style the table
for (row, col), cell in table.get_celld().items():
    if row == 0:  # Header row
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor('#333333')
    else:
        cell.set_text_props(color='white')
        cell.set_facecolor('#000000')
    cell.set_edgecolor('#666666')
    cell.set_linewidth(1)

# Highlight best model row
best_row_idx = models_list.index(best_model_enhanced) + 1
for j in range(len(headers)):
    table[(best_row_idx, j)].set_facecolor('#004d00')

ax9.set_title('Performance Summary Table', fontweight='bold', fontsize=14, color='white')

plt.tight_layout()
plt.subplots_adjust(top=0.93)  # Move plots closer to the title
save_path = f'models_results/{date.today().strftime("%Y-%m-%d_%H-%M")}_{global_epochs}e_{"500" if USE_FULL_DATASET else STOCK_LIMIT}s.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()